In [ ]:
import shutil

# Create the directory /kaggle/working/animal_dataset if it doesn't exist
!mkdir -p /kaggle/working/animal_dataset

# Copy entire dataset folder from Kaggle input to working directory
# dirs_exist_ok=True allows overwriting if destination folder already exists
shutil.copytree(
    "/kaggle/input/c2342342/Data/Kangaroo-Koala.v14-kangaroo-koala-v13.yolov8", 
    "/kaggle/working/animal_dataset", 
    dirs_exist_ok=True
)


In [ ]:
!pip install ultralytics

In [ ]:
import os
import cv2
import shutil
import albumentations as A

# --- 1. Define paths ---
image_dir = "/kaggle/working/animal_dataset/train/images"       # Original training images directory
label_dir = "/kaggle/working/animal_dataset/train/labels"       # Original YOLO label files directory
aug_img_dir = "/kaggle/working/animal_dataset/train/images_aug" # Directory to save augmented images
aug_lbl_dir = "/kaggle/working/animal_dataset/train/labels_aug" # Directory to save augmented labels

# Create directories for augmented images and labels if they don't exist
os.makedirs(aug_img_dir, exist_ok=True)
os.makedirs(aug_lbl_dir, exist_ok=True)

# --- 2. Define Albumentations augmentation pipeline ---
transform = A.Compose([
    A.HorizontalFlip(p=0.5),           # Flip images horizontally with 50% probability
    A.RandomBrightnessContrast(p=0.3),# Randomly adjust brightness and contrast with 30% probability
    A.Blur(blur_limit=3, p=0.2),      # Apply blur effect with blur kernel up to 3, 20% probability
    A.RandomGamma(p=0.3)               # Randomly adjust gamma with 30% probability
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))
# bbox_params tells Albumentations to expect bounding boxes in YOLO format (normalized x_center, y_center, width, height)
# and to associate 'class_labels' list to the bounding boxes during transformation

# --- 3. Function to validate bounding boxes ---
def is_valid_bbox(x, y, w, h):
    """
    Check if bounding box is valid:
    - Width and height must be larger than 0.01 (to avoid tiny boxes)
    - Box must be fully inside the image boundaries (normalized coordinates between 0 and 1)
    """
    return (w > 0.01 and h > 0.01 and
            x - w/2 >= 0 and x + w/2 <= 1 and
            y - h/2 >= 0 and y + h/2 <= 1)

# --- 4. Function to perform augmentation on a single image and save results ---
def augment_image(image_path, label_path, save_id):
    img = cv2.imread(image_path)           # Read original image using OpenCV
    h, w = img.shape[:2]                   # Get image height and width
    bboxes, class_labels = [], []         

    # Read bounding box annotations from YOLO label file
    with open(label_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            cls = int(parts[0])            # Class index
            x, y, bw, bh = map(float, parts[1:])  # bbox in YOLO format (normalized)
            if is_valid_bbox(x, y, bw, bh):
                bboxes.append([x, y, bw, bh])      # Collect valid bbox
                class_labels.append(str(cls))       # Collect corresponding class label

    # Skip if no valid bounding boxes in this image
    if not bboxes:
        return

    # Apply augmentation pipeline to image and bounding boxes
    try:
        transformed = transform(image=img, bboxes=bboxes, class_labels=class_labels)
    except Exception:
        # Skip image if augmentation breaks bbox validity
        return

    # Skip if all bounding boxes removed by augmentation
    if not transformed['bboxes']:
        return

    # Define output paths for augmented image and label file
    out_img = os.path.join(aug_img_dir, f"aug_{save_id}.jpg")
    out_lbl = os.path.join(aug_lbl_dir, f"aug_{save_id}.txt")

    # Save augmented image
    cv2.imwrite(out_img, transformed['image'])

    # Save augmented bounding box annotations in YOLO format
    with open(out_lbl, 'w') as f:
        for cls, box in zip(transformed['class_labels'], transformed['bboxes']):
            f.write(f"{cls} {box[0]:.6f} {box[1]:.6f} {box[2]:.6f} {box[3]:.6f}\n")

# --- 5. Loop through all label files and augment each corresponding image 3 times ---
save_id = 0
for fname in os.listdir(label_dir):
    if not fname.endswith(".txt"):
        continue
    img_path = os.path.join(image_dir, fname.replace(".txt", ".jpg"))
    lbl_path = os.path.join(label_dir, fname)
    if not os.path.exists(img_path):
        continue
    for _ in range(3):  # Augment each image 3 times
        augment_image(img_path, lbl_path, save_id)
        save_id += 1

print(f"Created {save_id} valid augmented images.")


In [ ]:
import shutil
import os

def merge_augmented_data(aug_img_dir, aug_lbl_dir, main_img_dir, main_lbl_dir):
    """
    Copy augmented images and labels from augmentation folders into the main training folders.

    Args:
        aug_img_dir (str): Directory path of augmented images.
        aug_lbl_dir (str): Directory path of augmented labels.
        main_img_dir (str): Directory path of the main training images.
        main_lbl_dir (str): Directory path of the main training labels.
    """
    img_count = 0
    
    # Copy each augmented image file to the main training images directory
    for f in os.listdir(aug_img_dir):
        shutil.copy(os.path.join(aug_img_dir, f), os.path.join(main_img_dir, f))
        img_count += 1
    
    # Copy each augmented label file to the main training labels directory
    for f in os.listdir(aug_lbl_dir):
        shutil.copy(os.path.join(aug_lbl_dir, f), os.path.join(main_lbl_dir, f))
    
    # Print summary of how many augmented images were merged
    print(f"Merged {img_count} augmented images into the training dataset.")

# Call the function to merge augmented data into the main dataset
merge_augmented_data(aug_img_dir, aug_lbl_dir, image_dir, label_dir)


In [ ]:
import os
import cv2
import numpy as np
import random
import shutil

# === Base directory and dataset paths ===
base_dir = "/kaggle/working/animal_dataset/train"
image_dir = os.path.join(base_dir, "images")    # Directory for training images
label_dir = os.path.join(base_dir, "labels")    # Directory for YOLO labels

# Output directories for augmented images and labels
mosaic_out = os.path.join(base_dir, "mosaic")
cutmix_out = os.path.join(base_dir, "cutmix")

# Create output directories if they don't exist
os.makedirs(mosaic_out + "/images", exist_ok=True)
os.makedirs(mosaic_out + "/labels", exist_ok=True)
os.makedirs(cutmix_out + "/images", exist_ok=True)
os.makedirs(cutmix_out + "/labels", exist_ok=True)

# --- Function to load YOLO labels and convert to bounding boxes in pixel coordinates ---
def load_yolo_label(label_path, img_w, img_h):
    """
    Load YOLO-format labels and convert to absolute pixel bbox coordinates.

    Args:
        label_path (str): Path to label file.
        img_w (int): Image width in pixels.
        img_h (int): Image height in pixels.

    Returns:
        boxes (list of list): Bounding boxes as [x1, y1, x2, y2] in pixels.
        classes (list of int): Corresponding class indices.
    """
    boxes, classes = [], []
    with open(label_path, 'r') as f:
        for line in f:
            cls, x, y, w, h = map(float, line.strip().split())
            if w <= 0 or h <= 0:  # Skip invalid boxes
                continue
            # Convert normalized YOLO format to pixel coordinates
            x1 = (x - w / 2) * img_w
            y1 = (y - h / 2) * img_h
            x2 = (x + w / 2) * img_w
            y2 = (y + h / 2) * img_h
            boxes.append([x1, y1, x2, y2])
            classes.append(int(cls))
    return boxes, classes

# --- Function to save bounding boxes in YOLO normalized format ---
def save_yolo_labels(path, boxes, classes, img_w, img_h):
    """
    Save bounding boxes and classes in YOLO normalized format to label file.

    Args:
        path (str): Output label file path.
        boxes (list): List of bounding boxes in pixel coordinates.
        classes (list): List of class indices.
        img_w (int): Image width.
        img_h (int): Image height.
    """
    with open(path, 'w') as f:
        for box, cls in zip(boxes, classes):
            x1, y1, x2, y2 = box
            # Convert pixel bbox to YOLO format (normalized x_center, y_center, width, height)
            x = ((x1 + x2) / 2) / img_w
            y = ((y1 + y2) / 2) / img_h
            w = (x2 - x1) / img_w
            h = (y2 - y1) / img_h
            # Skip invalid boxes
            if w <= 0 or h <= 0 or x < 0 or x > 1 or y < 0 or y > 1:
                continue
            f.write(f"{cls} {x:.6f} {y:.6f} {w:.6f} {h:.6f}\n")

# --- Mosaic augmentation function ---
def mosaic_augment(n_samples=10, out_size=640):
    """
    Create mosaic augmented images by combining 4 random images into one.

    Args:
        n_samples (int): Number of mosaic images to generate.
        out_size (int): Size to resize each sub-image.
    """
    image_files = [f for f in os.listdir(image_dir) if f.endswith('.jpg')]
    for i in range(n_samples):
        selected = random.sample(image_files, 4)
        # Create a blank mosaic image of size (2*out_size x 2*out_size)
        mosaic_img = np.full((out_size * 2, out_size * 2, 3), 114, dtype=np.uint8)
        labels, classes = [], []

        for idx, fname in enumerate(selected):
            img_path = os.path.join(image_dir, fname)
            lbl_path = os.path.join(label_dir, fname.replace(".jpg", ".txt"))
            img = cv2.imread(img_path)
            h, w = img.shape[:2]
            boxes, clss = load_yolo_label(lbl_path, w, h)
            img = cv2.resize(img, (out_size, out_size))
            x_off = (idx % 2) * out_size   # x offset (0 or out_size)
            y_off = (idx // 2) * out_size  # y offset (0 or out_size)
            mosaic_img[y_off:y_off+out_size, x_off:x_off+out_size] = img

            # Adjust boxes for mosaic offset and resize
            for box, cls in zip(boxes, clss):
                x1, y1, x2, y2 = box
                x1 = x1 * out_size / w + x_off
                y1 = y1 * out_size / h + y_off
                x2 = x2 * out_size / w + x_off
                y2 = y2 * out_size / h + y_off
                labels.append([x1, y1, x2, y2])
                classes.append(cls)

        out_img = os.path.join(mosaic_out, "images", f"mosaic_{i}.jpg")
        out_lbl = os.path.join(mosaic_out, "labels", f"mosaic_{i}.txt")
        cv2.imwrite(out_img, mosaic_img)
        save_yolo_labels(out_lbl, labels, classes, out_size * 2, out_size * 2)

# --- CutMix augmentation function ---
def cutmix_augment(n_samples=10, out_size=640):
    """
    Create cutmix augmented images by cutting and mixing patches of two images.

    Args:
        n_samples (int): Number of cutmix images to generate.
        out_size (int): Size to resize images.
    """
    image_files = [f for f in os.listdir(image_dir) if f.endswith('.jpg')]
    for i in range(n_samples):
        a, b = random.sample(image_files, 2)
        img_a = cv2.imread(os.path.join(image_dir, a))
        img_b = cv2.imread(os.path.join(image_dir, b))
        img_a = cv2.resize(img_a, (out_size, out_size))
        img_b = cv2.resize(img_b, (out_size, out_size))

        # Randomly choose a cut point in the image
        cut_x = random.randint(out_size//4, 3*out_size//4)
        cut_y = random.randint(out_size//4, 3*out_size//4)
        # Replace bottom-right patch of img_a with corresponding patch from img_b
        img_a[cut_y:, cut_x:] = img_b[cut_y:, cut_x:]

        labels, classes = [], []
        # Collect labels and classes from both images with position filtering for img_b
        for fname, offset_x, offset_y in [(a, 0, 0), (b, cut_x, cut_y)]:
            lbl_path = os.path.join(label_dir, fname.replace(".jpg", ".txt"))
            boxes, clss = load_yolo_label(lbl_path, out_size, out_size)
            for box, cls in zip(boxes, clss):
                x1, y1, x2, y2 = box
                cx = (x1 + x2) / 2
                cy = (y1 + y2) / 2
                # For second image b, only keep boxes in the replaced patch region
                if fname == b and (cx < cut_x or cy < cut_y):
                    continue
                labels.append([x1, y1, x2, y2])
                classes.append(cls)

        out_img = os.path.join(cutmix_out, "images", f"cutmix_{i}.jpg")
        out_lbl = os.path.join(cutmix_out, "labels", f"cutmix_{i}.txt")
        cv2.imwrite(out_img, img_a)
        save_yolo_labels(out_lbl, labels, classes, out_size, out_size)

# --- Function to merge augmented data into main dataset ---
def merge_augmented(aug_dir):
    """
    Copy augmented images and labels from augmented directory into main training folders.

    Args:
        aug_dir (str): Directory containing 'images' and 'labels' folders for augmented data.
    """
    # Copy images
    for f in os.listdir(os.path.join(aug_dir, "images")):
        shutil.copy(os.path.join(aug_dir, "images", f), os.path.join(image_dir, f))
    # Copy labels
    for f in os.listdir(os.path.join(aug_dir, "labels")):
        shutil.copy(os.path.join(aug_dir, "labels", f), os.path.join(label_dir, f))

# === Run augmentations ===
mosaic_augment(n_samples=20)    # Generate 20 mosaic augmented images
cutmix_augment(n_samples=20)    # Generate 20 cutmix augmented images

# === Merge augmented data into main dataset directories ===
merge_augmented(mosaic_out)
merge_augmented(cutmix_out)

print(" Mosaic + CutMix augmentation complete and merged.")


In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8x.pt")

# Train
model.train(
    data="/kaggle/working/animal_dataset/data.yaml",
    epochs=30,
    imgsz=640,
    batch=16,
    augment=True,
    workers=2,
    name="animal_aug_train",
    amp=True           # Mixed precision
)

In [ ]:
metrics = model.val()
print(metrics.box)  

In [ ]:
# Run model validation on the test split defined in the dataset YAML file
metrics = model.val(
    data="/kaggle/working/animal_dataset/data.yaml",
    split="test",      # Specify 'test' split if defined in the YAML
    imgsz=640          # Image size used for validation
)

# Access detection metrics from the results
precision = metrics.box.p.mean()     # Mean precision over all classes and IoU thresholds
recall = metrics.box.r.mean()        # Mean recall over all classes and IoU thresholds
map50 = metrics.box.map50            # Mean Average Precision at IoU=0.5
map_5095 = metrics.box.map           # Mean Average Precision averaged over IoU=0.5 to 0.95

# Calculate F1-score from precision and recall (with epsilon to avoid div by zero)
f1_score = 2 * (precision * recall) / (precision + recall + 1e-6)

# Print the evaluation metrics
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-score: {f1_score:.4f}")
print(f"mAP@0.5: {map50:.4f}")
print(f"mAP@0.5:0.95: {map_5095:.4f}")


In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
from collections import deque
import pandas as pd

# ======== Helper Classes ========
class SpeciesRiskConfig:
    def __init__(self, risk_dict=None):
        # Mapping species name to risk score
        self.risk_dict = risk_dict or {'kangaroo': 90, 'koala': 10}

    def score(self, name: str) -> int:
        # Return risk score for given species name (case insensitive)
        return self.risk_dict.get(name.lower(), 0)

class RoadAreaChecker:
    def __init__(self, polygon_points, img_w, img_h):
        # Create boolean mask of road area polygon on the image
        self.mask = self._poly_to_mask(polygon_points, img_w, img_h)

    def _poly_to_mask(self, pts, w, h):
        # Generate a mask from polygon points
        mask = np.zeros((h, w), dtype=np.uint8)
        cv2.fillPoly(mask, [np.array(pts, dtype=np.int32)], 1)
        return mask.astype(bool)

    def is_near_road(self, bbox):
        # Check if bounding box overlaps with road area mask
        x1, y1, x2, y2 = map(int, bbox)
        x1, y1 = max(x1, 0), max(y1, 0)
        x2, y2 = min(x2, self.mask.shape[1]-1), min(y2, self.mask.shape[0]-1)
        roi = self.mask[y1:y2+1, x1:x2+1]
        return roi.any()

class DistanceEstimator:
    def __init__(self, img_h, near_ratio=0.4):
        # Initialize with image height and ratio threshold for "close" object
        self.img_h = img_h
        self.near_ratio = near_ratio

    def is_close(self, bbox):
        # Determine if object height relative to image height exceeds near_ratio
        x1, y1, x2, y2 = bbox
        box_h = y2 - y1
        return (box_h / self.img_h) >= self.near_ratio

class DayNightDetector:
    def __init__(self, threshold=60):
        # Threshold for mean luminance to distinguish day/night
        self.threshold = threshold

    def get_time_of_day(self, img_bgr):
        # Convert BGR image to YCrCb and compute mean luminance channel (Y)
        y_cr_cb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2YCrCb)
        y_mean  = y_cr_cb[..., 0].mean()
        return 'night' if y_mean < self.threshold else 'day'

class RiskSmoother:
    def __init__(self, window=5, hi_thresh=0.6):
        # Maintain buffer of risk levels over last 'window' frames
        self.window = window
        self.hi_thresh = hi_thresh
        self.buffer = deque(maxlen=window)

    def update(self, risk_lvl: str) -> str:
        # Update buffer and determine smoothed risk level
        self.buffer.append(risk_lvl)
        if len(self.buffer) < self.window:
            return risk_lvl
        high_ratio = sum(r=='HIGH' for r in self.buffer)/self.window
        return 'HIGH' if high_ratio >= self.hi_thresh else self.buffer[-1]

# ======== Main Agent Class: ThreatAssessmentAgentV2 ========
class ThreatAssessmentAgentV2:
    def __init__(self,
                 species_risk: SpeciesRiskConfig,
                 road_checker : RoadAreaChecker,
                 dist_est     : DistanceEstimator,
                 day_night    : DayNightDetector,
                 smoother     : RiskSmoother,
                 high_score=70,
                 med_score=40,
                 conf_thr=0.6):
        # Initialize agent with all components and thresholds
        self.species_risk  = species_risk
        self.road_checker  = road_checker
        self.dist_est      = dist_est
        self.day_night_det = day_night
        self.smoother      = smoother
        self.high_score    = high_score
        self.med_score     = med_score
        self.conf_thr      = conf_thr

    def assess_risk(self, img_bgr, det):
        # Assess risk of a detected object based on multiple criteria
        cls   = det['class_name'].lower()
        conf  = det['confidence']
        bbox  = det['bbox']
        tod = det.get('time_of_day') or self.day_night_det.get_time_of_day(img_bgr)
        score = self.species_risk.score(cls)
        near_road = self.road_checker.is_near_road(bbox)
        close_obj = self.dist_est.is_close(bbox)

        risk = 'LOW'  # Default risk
        if score >= self.high_score and (near_road or close_obj):
            risk = 'HIGH'
        elif (score >= self.med_score and tod == 'night' and conf >= self.conf_thr):
            risk = 'MEDIUM'
        return self.smoother.update(risk)

# ======== Configuration of paths and model loading ========
model_path = "/kaggle/working/runs/detect/animal_aug_train/weights/best.pt"
image_path = "/kaggle/working/animal_dataset/test/images/00001_jpg.rf.a47ad67a15d9815ec15eaa4ebbb936ef.jpg"

img = cv2.imread(image_path)
H, W = img.shape[:2]
model = YOLO(model_path)
results = model(img)[0]  # Get first result from YOLO predictions

# ======== Instantiate helper classes and agent ========
species_cfg = SpeciesRiskConfig({'kangaroo': 90, 'koala': 10})
road_poly   = [(0, H-130), (W, H-130), (W, H), (0, H)]  # Road polygon area at bottom of image
road_chk    = RoadAreaChecker(road_poly, img_w=W, img_h=H)
dist_est    = DistanceEstimator(img_h=H, near_ratio=0.45)
daynight    = DayNightDetector(threshold=70)
smoother    = RiskSmoother(window=5, hi_thresh=0.6)

agent = ThreatAssessmentAgentV2(
    species_risk=species_cfg,
    road_checker=road_chk,
    dist_est=dist_est,
    day_night=daynight,
    smoother=smoother,
    high_score=70,
    med_score=40,
    conf_thr=0.6
)

# ======== Risk assessment for detected objects ========
outputs = []
for r in results.boxes:
    det = {
        'class_name' : model.names[int(r.cls)],
        'confidence' : float(r.conf),
        'bbox': r.xyxy[0].tolist(),  # Extract bounding box coordinates as list
    }
    risk = agent.assess_risk(img, det)
    outputs.append({
        'class'     : det['class_name'],
        'confidence': det['confidence'],
        'bbox'      : det['bbox'],
        'risk'      : risk
    })

# ======== Display results in a DataFrame ========
df = pd.DataFrame(outputs)
display(df)


In [ ]:
class CommunicationAgent:
    def __init__(self, enable_mobile_alert=False):
        # Initialize communication agent with optional mobile alert enabled or disabled
        self.enable_mobile_alert = enable_mobile_alert

    def generate_message(self, risk_level):
        # Generate alert message based on the risk level
        if risk_level == "HIGH":
            return " [CRITICAL] HIGH risk detected! Immediate action required."
        elif risk_level == "MEDIUM":
            return " [WARNING] Medium risk detected. Monitor the situation."
        else:
            return " [SAFE] No significant threat detected."

    def send_alert(self, message):
        # Send alert message to road display system (console print here)
        print(" [ROAD DISPLAY SYSTEM]")
        print(message)
        # Optionally send alert to mobile app notification
        if self.enable_mobile_alert:
            print(" [MOBILE APP NOTIFICATION]")
            print(message)

    def handle_risk(self, risk_level):
        # Handle risk by generating and sending alerts for HIGH and MEDIUM levels,
        # or print status message for LOW risk
        if risk_level in ["HIGH", "MEDIUM"]:
            msg = self.generate_message(risk_level)
            self.send_alert(msg)
        else:
            print(" [STATUS] LOW risk - no alert needed.")

# Example usage
risk_level = "HIGH"  # Can be "HIGH", "MEDIUM", or "LOW"
comm_agent = CommunicationAgent(enable_mobile_alert=True)
comm_agent.handle_risk(risk_level)


In [ ]:
import datetime

class MonitoringAgent:
    def __init__(self, log_to_file=True, log_path="monitoring_log.txt"):
        # Initialize the monitoring agent with options to log to file or console
        self.log_to_file = log_to_file
        self.log_path = log_path

    def _write(self, text):
        # Write a timestamped log line either to file or print to console
        timestamp = datetime.datetime.now().strftime("[%Y-%m-%d %H:%M:%S]")
        line = f"{timestamp} {text}"
        if self.log_to_file:
            with open(self.log_path, "a") as f:
                f.write(line + "\n")
        else:
            print(line)

    def log_detection_event(self, detections):
        # Log details of detected objects (class, confidence, bounding box)
        self._write("📦 Detection Event:")
        for det in detections:
            self._write(f" - {det['class']} (conf={det['confidence']:.2f}) at {det['bbox']}")

    def log_risk_assessment(self, assessed):
        # Log risk assessment results for each detected object
        self._write("🧠 Risk Assessment:")
        for obj in assessed:
            self._write(f" - {obj['class']} → Risk = {obj['risk']}")

    def log_alert_status(self, risk_level):
        # Log whether an alert was issued based on overall risk level
        alert = "YES" if risk_level in ["HIGH", "MEDIUM"] else "NO"
        self._write(f"🚨 Alert Issued: {alert} (Overall Risk = {risk_level})")

# Example usage:
final_risk = "HIGH"  

# Create monitoring agent instance with file logging enabled
monitor = MonitoringAgent(log_to_file=True)  

# Log detected objects info
monitor.log_detection_event(outputs)

# Log risk assessments of detected objects
monitor.log_risk_assessment(outputs)

# Log whether alert was issued for the final risk level
monitor.log_alert_status(final_risk)


In [ ]:
# Predict on test images using the YOLO model
results = model.predict(
    source="/kaggle/working/animal_dataset/test/images",  # Folder containing test images
    save=False,    # Do not save the predicted images with bounding boxes
    conf=0.25      # Confidence threshold for detections
)

# Count total number of bounding boxes detected across all test images
total_boxes = sum(len(r.boxes) for r in results)

print(f"\nBounding boxes in the test images: {total_boxes}")


In [ ]:
from ultralytics import YOLO

# Load the trained YOLO model (you can replace path with your trained weights or yolov8.pt)
model = YOLO("/kaggle/working/runs/detect/animal_aug_train/weights/best.pt")

# Predict on the augmented training images folder
results = model.predict(
    source="/kaggle/working/animal_dataset/train/images_aug",  # Augmented train images path
    conf=0.25,    # Confidence threshold
    save=False    # Do not save prediction images with bounding boxes
)

# Count total predicted bounding boxes across all augmented training images
total_pred_boxes = sum(len(r.boxes) for r in results)

print(f"🔢 Total YOLO predicted bounding boxes on augmented training set: {total_pred_boxes}")


In [ ]:
from collections import Counter

# Initialize counter to keep track of predicted bounding boxes per class
box_class_counter = Counter()

# Iterate over each prediction result (for each image)
for r in results:
    # Iterate over detected class IDs in current image
    for c in r.boxes.cls:
        cls_id = int(c.item())
        box_class_counter[cls_id] += 1

# Print the count of predicted bounding boxes grouped by class
print("\n📊 Number of predicted bounding boxes per class:")
for cls_id, count in box_class_counter.items():
    print(f"  Class {cls_id} ({model.names[cls_id]}): {count}")
